In [1]:
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

drivers = pd.read_csv(
    "../data/drivers.csv",
    sep=",",
    thousands=None,
    decimal=".",
)

timesheet = pd.read_csv(
    "../data/timesheet.csv",
    sep=",",
    thousands=None,
    decimal=".",
)

connection = sqlite3.connect("../temp/db.sqlite")

connection.executescript(
    """
    DROP VIEW IF EXISTS driver_summary;
    DROP VIEW IF EXISTS drivers;
    DROP VIEW IF EXISTS timesheet;
    """
)

drivers.to_sql("drivers_raw", connection, if_exists="replace", index=False)
timesheet.to_sql("timesheet_raw", connection, if_exists="replace", index=False)

connection.executescript(
    """
    CREATE VIEW drivers AS
    SELECT
        driverId AS driver_id,
        name,
        ssn,
        location,
        certified,
        "wage-plan" AS wage_plan
    FROM drivers_raw;

    CREATE VIEW timesheet AS
    SELECT
        driverId AS driver_id,
        week,
        "hours-logged" AS hours_logged,
        "miles-logged" AS miles_logged
    FROM timesheet_raw;

    CREATE VIEW driver_summary AS
    SELECT
        t.driver_id,
        SUM(t.hours_logged) AS hours_logged,
        SUM(t.miles_logged) AS miles_logged,
        d.name
    FROM timesheet AS t
    INNER JOIN drivers AS d
        ON t.driver_id = d.driver_id
    GROUP BY
        t.driver_id,
        d.name;
    """
)

connection.commit()

sql = """
SELECT
    driver_id,
    hours_logged,
    miles_logged,
    name
FROM driver_summary
ORDER BY driver_id;
"""

summary = pd.read_sql_query(sql, connection)

summary.to_csv(
    "../submission/summary.csv",
    sep=",",
    header=True,
    index=False,
)

sql = """
SELECT
    driver_id,
    hours_logged,
    miles_logged,
    name
FROM driver_summary
ORDER BY miles_logged DESC
LIMIT 10;
"""

top10 = pd.read_sql_query(sql, connection)

plt.figure(figsize=(10, 6))
plt.barh(top10["name"], top10["miles_logged"])
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../submission/top10_drivers.png", bbox_inches="tight")
plt.close()

connection.close()